In [41]:
import os
import re

import dotenv
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

dotenv.load_dotenv()

DOCS_CHUNK_SIZE = int(os.getenv("DOCS_CHUNK_SIZE"))
DOCS_CHUNK_OVERLAP = int(os.getenv("DOCS_CHUNK_OVERLAP"))
ALUMNO = os.getenv("ALUMNO")


# Trabajo Práctico 3

Enunciado:

Implementar un sistema de agentes para que responda de manera eficiente dependiendo de qué persona se está preguntando (1 agente por persona)

Por defecto, cuando no se nombra a nadie en la query, utilizar el Agente del alumno.

Atención: Si se consulta por más de un CV, traer el contexto de cada uno y
responder de manera acorde

## Entidades

Crearé dos clases que representaran los distintos agentes del trabajo.

Agent: serán los agentes focalizados en curriculums de una sola persona

Multiagent: Sólo habrá una instancia de esta clase y actuará cuando se quiera comparar curriculum de más de una persona 

In [61]:
class Agent:
    def __init__(self, filename: str, filepath: str, embeddings, llm):
        name = os.path.splitext(filename)[0].lower()
        self.name = name
        self.cv_filepath = filepath
        self.embeddings = embeddings
        self.index = name
        self.llm = llm
        self.default = name == ALUMNO

        self.cv = None
        self.splits = None
        self.vectorstore = None
        self.retriever = None
        self.chain = None

        self._load_cv()
        self._create_vectorstore()
        self._create_chain()

    def _load_cv(self):
        loader = PyPDFLoader(self.cv_filepath)
        self.cv = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=DOCS_CHUNK_SIZE, chunk_overlap=DOCS_CHUNK_OVERLAP)
        self.chunks = splitter.split_documents(self.cv)
        self.vectors = embeddings.embed_documents([doc.page_content for doc in self.chunks])

    def _create_vectorstore(self):
        self.vectorstore = PineconeVectorStore.from_documents(
            self.chunks,
            self.embeddings,
            index_name=self.index
        )
        self.retriever = self.vectorstore.as_retriever()

    def _create_chain(self):
        prompt = ChatPromptTemplate.from_template(
            "Usa el siguiente CV de {name} para responder la pregunta.\n"
            "Contexto:\n{context}\n\n"
            "Pregunta: {input}"
        )

        doc_chain = create_stuff_documents_chain(self.llm, prompt)
        self.chain = create_retrieval_chain(self.retriever, doc_chain)

    def answer(self, question: str, history) -> str:
        result = self.chain.invoke({"input": question, "name": self.name, "chat_history": history})
        return result["answer"]

In [43]:
class MultiAgent:
    def __init__(self, agents, llm, name):
        self.agents = agents
        self.llm = llm
        self.name = name

    def answer(self, mentioned, question: str):
        answers = []
        for agent in mentioned:
            result = agent.answer(question, conversation_history)
            answers.append({
                "agent": agent.name,
                "answer": result,
            })

        context = "\n\n".join(
            [f"[{a['agent'].capitalize()}]: {a['answer']}" for a in answers]
        )

        prompt = f"""La consulta fue: {question}

        Aquí tienes información de varios CVs:
        {context}

        Responde de forma integrada y comparativa, mencionando a cada persona según corresponda.
        """

        return self.llm.invoke(prompt).content


## Integraciones

Utilizaré embeddings de HuggingFace y como LLM a Groq. Para la base de datos vectorial utilizaré Pinecone

In [44]:
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDINGS_MODEL"))
groq = ChatGroq(model=os.getenv("GROQ_MODEL"), temperature=0)

Crearé agenes igual a la cantidad de archivos pdf en la carpeta docs

In [45]:
def create_agents_from_cvs(directory="docs"):
    agents = []
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            filepath = os.path.join(directory, filename)
            agent = Agent(filename, filepath, embeddings=embeddings, llm=groq)
            agents.append(agent)

    return agents

In [47]:
agents = create_agents_from_cvs("docs")
multiagent = MultiAgent(agents, groq, "multiagent")

In [48]:
pinecone = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
spec = ServerlessSpec(cloud=os.getenv("PINECONE_CLOUD"), region=os.getenv("PINECONE_REGION"))

## Base de datos

Se crearán los indices de cada una de las personas de los curriculums en Pinecone. Luego se subirán los vectores de embeddings a Pinecone a cada uno de los indices

In [49]:
def recreate_index(index_name, pinecone, spec):
    if index_name in pinecone.list_indexes().names():
        pinecone.delete_index(index_name)
    print("index {} borrado".format(index_name))

    if index_name not in pinecone.list_indexes().names():
        print("index creado con el nombre: {}".format(index_name))
        pinecone.create_index(
            index_name,
            dimension=384,
            metric='cosine',
            spec=spec
        )
    else:
        print("el index con el nombre {} ya estaba creado".format(index_name))

In [50]:
def upload_to_pinecone(index, documents, vectors):
    for i, (doc, vector) in enumerate(zip(documents, vectors)):
        pinecone_index = pinecone.Index(index)
        pinecone_index.upsert([
            (
                f"chunk-{i}",
                vector,
                {"text": doc.page_content}
            )
        ])

    print(f"vectores cargados en {index}")

In [51]:
for agent in agents:
    recreate_index(agent.index, pinecone, spec)
    upload_to_pinecone(agent.index, agent.chunks, agent.vectors)

index christian borrado
index creado con el nombre: christian
vectores cargados en christian
index joaquin borrado
index creado con el nombre: joaquin
vectores cargados en joaquin
index javier borrado
index creado con el nombre: javier
vectores cargados en javier


## Conversaciones 

### Función Ask

Se define un método ask qué decidirá qué agente debe responder las preguntas:

In [52]:
def ask(question: str, agents, conversation_history=[]):
    mentioned = [
        agent for agent in agents
        if re.search(agent.name, question, re.IGNORECASE)
    ]

    if len(mentioned) == 0:
        queried_agent = next(agent for agent in agents if agent.default)
        answer = queried_agent.answer(question, conversation_history)

    elif len(mentioned) == 1:
        queried_agent = mentioned[0]
        answer = queried_agent.answer(question, conversation_history)

    else:
        queried_agent = multiagent
        answer = queried_agent.answer(mentioned, question)

    print(f"[{queried_agent.name.capitalize()}]: {answer}")


### Ejemplos

A continuación diversos ejemplos de conversaciones:

In [53]:
conversation_history = []

In [56]:
ask("¿Qué experiencia tiene Christian en IA?", agents, conversation_history)

[Christian]: Según el CV de Christian, su experiencia en Inteligencia Artificial (IA) se limita a su trabajo en Navent, donde diseñó y desarrolló aplicaciones de Data Mining en C# utilizando tecnologías como MySQL, Selenium Web Driver y Fiddler. También desarrolló un Data Warehouse en MySQL utilizando Pentaho Data Integration como herramienta ETL.

Aunque no se menciona explícitamente la Inteligencia Artificial, el Data Mining es un campo relacionado con la IA que implica el análisis y la extracción de patrones y conocimientos a partir de grandes conjuntos de datos. Por lo tanto, se puede inferir que Christian tiene experiencia en áreas relacionadas con la IA, aunque no se especifica si ha trabajado directamente en proyectos de IA.

En resumen, la experiencia de Christian en IA se limita a su trabajo en Data Mining y desarrollo de Data Warehouse, pero no se menciona experiencia directa en áreas como el aprendizaje automático, el procesamiento de lenguaje natural o la visión artificial,

In [55]:
ask("¿Qué experiencia tiene en IA?", agents, conversation_history)


[Christian]: Según el CV de Christian, tiene experiencia en el desarrollo de aplicaciones de Data Mining utilizando C# y MySQL, lo que sugiere que tiene experiencia en el campo de la Inteligencia Artificial (IA) relacionada con el análisis de datos y la minería de datos. Además, menciona el uso de herramientas como Selenium Web Driver y Fiddler para la extracción de datos de la web, lo que también se relaciona con la IA.

En particular, su experiencia en Navent (08/2015 - Presente) incluye:

* Diseño y desarrollo de aplicaciones de Data Mining utilizando C# y MySQL
* Uso de Selenium Web Driver y Fiddler para la extracción de datos de la web
* Desarrollo de un Data Warehouse en MySQL utilizando Pentaho Data Integration como herramienta ETL

Sin embargo, no hay menciones explícitas de experiencia en áreas de IA como el aprendizaje automático, el procesamiento de lenguaje natural o la visión artificial. Por lo tanto, su experiencia en IA parece estar enfocada en la minería de datos y el a

In [59]:
ask("¿Qué experiencia tiene Javier en IA?", agents, conversation_history)


[Javier]: Según el CV proporcionado, no hay menciones a experiencia en Inteligencia Artificial (IA) de Javier. Su experiencia y formación se centran en la música, específicamente en el contrabajo y la docencia en instituciones musicales y educativas.


In [60]:
ask("¿Quien sabe más de música Javier o Joaquin?", agents, conversation_history)


[Multiagent]: Basándome en la información proporcionada, puedo concluir que Javier tiene una amplia experiencia y formación en música, lo que sugiere que sabe más de música que Joaquín. Mientras que el CV de Javier destaca su formación en el Conservatorio Superior de Música de la Ciudad de Buenos Aires Astor Piazzolla, su participación en orquestas sinfónicas y su trabajo con maestros reconocidos en el campo de la música, el CV de Joaquín no menciona la música como una de sus habilidades o intereses.

En comparación, la experiencia y formación de Javier en música son significativas, ya que ha estudiado en instituciones prestigiosas y ha trabajado con profesionales destacados en el campo. Por otro lado, no hay información disponible que sugiera que Joaquín tenga conocimientos o experiencia en música, lo que hace que sea difícil comparar su nivel de conocimiento con el de Javier.

En resumen, considerando la información disponible, Javier parece tener una ventaja significativa en término